# Python OOP — Encapsulation
### Study notes • Beginner-friendly • Examples and practice

## Learning goals
By the end of these notes, you should be able to:
- Explain encapsulation and why it is useful.
- Distinguish public, single-underscore, and double-underscore names.
- Explain Python's name-mangling behavior.
- Use getters, setters, and `@property`.
- Apply encapsulation to a practical class.

> **Core idea:** Encapsulation brings data and the methods that work with it together in a class, and provides a deliberate interface for interacting with an object's state.

## 1. Why do we need encapsulation?

Imagine a bank account whose balance can be changed to any value:

```python
account.balance = -5000
```

If a negative balance is not allowed by the rules of our application, unrestricted assignment makes it easy to put the object into an invalid state.

Encapsulation helps us design a class that offers suitable operations—such as `deposit()` and `withdraw()`—and validates changes where necessary.

It is not simply about hiding variables. It is also about **controlling state and keeping an object's data valid**.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        if balance < 0:
            raise ValueError("Initial balance cannot be negative")
        self.balance = balance


account = BankAccount(1000)
print(account.balance)

# This is allowed because balance is public:
account.balance = -5000
print(account.balance)

The example validates the initial balance, but it does not validate later assignments. We can improve the interface by storing the balance internally and exposing controlled operations.

## 2. Public, internal-by-convention, and name-mangled attributes

Python commonly uses these naming styles:

| Style | Example | Meaning |
|---|---|---|
| Public | `name` | Intended to be part of the public interface |
| Internal by convention | `_name` | Intended for internal use; avoid relying on it externally |
| Name-mangled | `__name` | Python transforms the name to help avoid accidental name clashes |

### Important
A single underscore is **only a convention**. Python does not block external access to `_name`.

A double leading underscore triggers **name mangling**. It is not a security mechanism and does not make an attribute absolutely inaccessible.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name        # Public
        self._section = "CSE"   # Internal-use convention
        self.__marks = marks   # Name-mangled


student = Student("Sadik", 85)

print(student.name)
print(student._section)  # Allowed, but signals internal use

# print(student.__marks)  # AttributeError: original name is not present
print(student._Student__marks)  # Works, because of name mangling
print(student.__dict__)

### What is name mangling?

Inside a class named `Student`, an attribute written as `self.__marks` is transformed to a name like `self._Student__marks`.

This helps prevent accidental name conflicts, particularly when a class is subclassed. It does not prevent deliberate access: code can still use the mangled name.

**Do not rely on underscores to protect passwords, API keys, or other secrets from code that can inspect the Python process.**

## 3. A single underscore: a signal, not a restriction

```python
self.name
self._marks
```

For ordinary attribute access, these behave alike: both can be read and assigned from outside the class.

The difference is the intention communicated to other programmers:

- `name`: intended as public.
- `_marks`: intended as an implementation detail; external code should generally avoid depending on it.

Single-underscore names also receive special treatment in wildcard imports: `from module import *` normally excludes names beginning with `_`, unless the module defines `__all__`.

## 4. Getters and setters

A **getter** retrieves a value. A **setter** changes a value, often after validation.

The traditional method-based style looks like this:

In [ ]:
class Student:
    def __init__(self, marks):
        self.__marks = marks

    def get_marks(self):
        return self.__marks

    def set_marks(self, value):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise TypeError("Marks must be numeric")
        if not 0 <= value <= 100:
            raise ValueError("Marks must be between 0 and 100")
        self.__marks = value


student = Student(85)
print(student.get_marks())

student.set_marks(95)
print(student.get_marks())

This works, but Python often uses `@property` to make controlled access feel like ordinary attribute access.

## 5. `@property`: the Pythonic approach

- `@property` defines a getter accessed like `obj.marks`.
- `@marks.setter` defines what happens when code assigns `obj.marks = value`.
- Store the actual value under a different name, such as `self.__marks`, to avoid recursively calling the same property.

In [ ]:
class Student:
    def __init__(self, marks):
        self.marks = marks  # Calls the setter, so initial marks are validated too.

    @property
    def marks(self):
        return self.__marks

    @marks.setter
    def marks(self, value):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise TypeError("Marks must be numeric")
        if not 0 <= value <= 100:
            raise ValueError("Marks must be between 0 and 100")
        self.__marks = value


student = Student(85)
print(student.marks)

student.marks = 95
print(student.marks)

# Uncomment to test validation:
# student.marks = 150  # ValueError
# student.marks = "high"  # TypeError

### Why not assign `self.marks = value` inside the setter?

That assignment calls the same setter again, which calls itself repeatedly until Python raises `RecursionError`.

Incorrect:
```python
@marks.setter
def marks(self, value):
    self.marks = value
```

Correct:
```python
@marks.setter
def marks(self, value):
    self.__marks = value
```

The correct version stores the value in the underlying attribute after validating it.

## 6. Read-only properties

A property with no setter can be read, but ordinary assignment to it raises `AttributeError`.

Properties are also useful for calculated values that do not need to be stored separately.

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height


rectangle = Rectangle(10, 5)
print(rectangle.area)  # 50

# rectangle.area = 100  # AttributeError: no setter

Here, `area` is calculated from `width` and `height`. It is not stored as a separate attribute, so it cannot become stale when the dimensions change.

A read-only property does not automatically make the attributes it depends on read-only.

## 7. Encapsulation and inheritance

A subclass inherits a parent's attributes, but name mangling can surprise beginners.

An attribute `self.__value` written in `Parent` is mangled to `_Parent__value`. Writing `self.__value` inside `Child` instead refers to `_Child__value`, which is a different name.

In [ ]:
class Parent:
    def __init__(self):
        self.__value = 100

    def get_value(self):
        return self.__value


class Child(Parent):
    def show(self):
        print(self.get_value())


obj = Child()
obj.show()  # 100

The child uses the parent's public method rather than depending on the parent's mangled attribute name.

`super()` is used to call behavior according to the method resolution order; it does not disable name mangling or grant special access to private-style attributes.

In [ ]:
class Camera:
    def __init__(self, megapixels):
        self.__megapixels = megapixels

    def show_camera(self):
        print(self.__megapixels)


class Phone(Camera):
    def __init__(self, megapixels, brand):
        super().__init__(megapixels)
        self.brand = brand


phone = Phone(50, "Samsung")
phone.show_camera()  # 50
print(phone.brand)   # Samsung

## 8. Practical example: a bank account

This example combines internal state, a read-only property, and validated operations.

It is an educational example, not a complete production banking system.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if not isinstance(balance, (int, float)) or isinstance(balance, bool):
            raise TypeError("Balance must be numeric")
        if balance < 0:
            raise ValueError("Initial balance cannot be negative")

        self.owner = owner
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise TypeError("Deposit must be numeric")
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise TypeError("Withdrawal must be numeric")
        if amount <= 0:
            raise ValueError("Withdrawal must be positive")
        if amount > self.__balance:
            raise ValueError("Insufficient balance")
        self.__balance -= amount


account = BankAccount("Sadik", 1000)
account.deposit(500)
account.withdraw(200)

print(account.owner)    # Sadik
print(account.balance)  # 1300

# account.balance = 50000  # AttributeError: balance is read-only

### Practical design and security notes

- Use `Decimal`, rather than binary floating-point numbers, for real monetary calculations.
- Validate inputs at the class boundary.
- Do not treat `_name` or `__name` as a security boundary.
- In a real financial system, authorization, persistence, audit logs, and concurrency controls are also necessary.
- Do not add getters and setters to every attribute automatically. Use them when they provide meaningful behavior or a stable interface.

## 9. Common mistakes

### Mistake A: Thinking a single underscore prevents access
```python
self._value = 10
obj._value  # Still accessible
```

### Mistake B: Thinking double underscore means absolute privacy
```python
self.__value = 10
obj._ClassName__value  # Can still be accessed deliberately
```

### Mistake C: Skipping validation in the constructor
If the constructor directly assigns `self.__marks = marks`, the property setter is not called. Use `self.marks = marks` when you want the setter's validation to apply.

### Mistake D: Recursion in a setter
Assign to the backing attribute (`self.__marks`), not to the property (`self.marks`), inside its own setter.

### Mistake E: Making every attribute private
Encapsulation is about a thoughtful interface, not about adding underscores everywhere.

## 10. Quick revision table

| Concept | Meaning |
|---|---|
| Encapsulation | Organize data and behavior, and control how object state is accessed or changed |
| `name` | Public by convention |
| `_name` | Internal use by convention; no access restriction |
| `__name` | Name mangling helps avoid accidental name clashes |
| Getter | Retrieves a value |
| Setter | Updates a value, often after validation |
| `@property` | Lets a method be accessed with attribute syntax |
| Read-only property | Property with no setter |
| `raise ValueError(...)` | Signals an invalid value |
| `super()` | Calls behavior through the method resolution order |

### Five key takeaways

1. Encapsulation helps keep an object's state valid.
2. A single underscore is a signal to other programmers, not a restriction.
3. Double underscores trigger name mangling, not absolute privacy.
4. Properties support validation while keeping natural attribute syntax.
5. Encapsulation is useful for design, but it is not a security boundary.

## 11. Practice exercises

Try these in a fresh code cell before checking your own answers.

### Exercise 1 — Name mangling
Create a class `Example` with `x`, `_y`, and `__z`. Check which attributes can be accessed externally and inspect `obj.__dict__`.

### Exercise 2 — Validated student
Create a `Student` class with a `marks` property. Marks must be numeric and between 0 and 100. Validate both initial values and later assignments.

### Exercise 3 — Inheritance
Create `Parent` with a double-underscore attribute and a method that returns it. Create `Child(Parent)` and access the value through the parent method.

### Exercise 4 — Bank account
Implement `deposit()` and `withdraw()` with validation. Make `balance` a read-only property.

### Self-check questions
1. Is `_value` inaccessible outside a class?
2. What name does `__value` become inside a class called `Example`?
3. Why should a setter assign to a backing attribute rather than to its own property?
4. Why might a read-only property be useful?
5. Does name mangling protect secrets from malicious code in the same Python process?